# Customer Segmentation Analysis

## Objective
Segment customers based on their behavior and value to enable targeted marketing and personalized customer experiences.

## Methodology:
1. **RFM Analysis** - Recency, Frequency, Monetary segmentation
2. **K-Means Clustering** - Machine learning-based segmentation
3. **Customer Profiling** - Demographic and behavioral insights
4. **Actionable Recommendations** - Marketing strategies per segment

In [ ]:
# Import libraries
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from datetime import datetime, timedelta
from sklearn.preprocessing import StandardScaler
from sklearn.cluster import KMeans
from sklearn.metrics import silhouette_score
import warnings
warnings.filterwarnings('ignore')

# Settings
plt.style.use('seaborn-v0_8-whitegrid')
sns.set_palette('Set2')
pd.set_option('display.max_columns', None)
pd.set_option('display.float_format', '{:.2f}'.format)

print("Libraries loaded successfully!")

## 1. Data Loading and Preparation

In [ ]:
# Load data
sales_df = pd.read_csv('../data/sales_data.csv')
customer_df = pd.read_csv('../data/customer_data.csv')

# Convert dates
sales_df['transaction_date'] = pd.to_datetime(sales_df['transaction_date'])
customer_df['registration_date'] = pd.to_datetime(customer_df['registration_date'])

print(f"Sales records: {len(sales_df):,}")
print(f"Unique customers: {sales_df['customer_id'].nunique():,}")
print(f"Customer database: {len(customer_df):,}")

In [ ]:
# Define analysis date (most recent transaction date)
analysis_date = sales_df['transaction_date'].max()
print(f"Analysis Date: {analysis_date}")

## 2. RFM Analysis

RFM stands for:
- **Recency**: How recently did the customer make a purchase?
- **Frequency**: How often do they purchase?
- **Monetary**: How much do they spend?

In [ ]:
# Calculate RFM metrics for each customer
rfm_df = sales_df.groupby('customer_id').agg({
    'transaction_date': lambda x: (analysis_date - x.max()).days,  # Recency
    'transaction_id': 'nunique',  # Frequency
    'revenue': 'sum'  # Monetary
}).reset_index()

rfm_df.columns = ['customer_id', 'recency', 'frequency', 'monetary']

print("RFM Metrics Calculated:")
display(rfm_df.head(10))
print("\nRFM Statistics:")
display(rfm_df.describe())

In [ ]:
# Create RFM scores (1-5 scale, 5 being best)
rfm_df['r_score'] = pd.qcut(rfm_df['recency'], 5, labels=[5, 4, 3, 2, 1])  # Lower recency is better
rfm_df['f_score'] = pd.qcut(rfm_df['frequency'].rank(method='first'), 5, labels=[1, 2, 3, 4, 5])
rfm_df['m_score'] = pd.qcut(rfm_df['monetary'], 5, labels=[1, 2, 3, 4, 5])

# Convert to numeric
rfm_df['r_score'] = rfm_df['r_score'].astype(int)
rfm_df['f_score'] = rfm_df['f_score'].astype(int)
rfm_df['m_score'] = rfm_df['m_score'].astype(int)

# Calculate total RFM score
rfm_df['rfm_score'] = rfm_df['r_score'] + rfm_df['f_score'] + rfm_df['m_score']

print("RFM Scores:")
display(rfm_df.head(10))

In [ ]:
# Define customer segments based on RFM scores
def rfm_segment(row):
    if row['rfm_score'] >= 13:
        return 'Champions'
    elif row['rfm_score'] >= 11:
        return 'Loyal Customers'
    elif row['r_score'] >= 4 and row['f_score'] <= 2:
        return 'Recent Customers'
    elif row['f_score'] >= 4:
        return 'Frequent Buyers'
    elif row['m_score'] >= 4:
        return 'High Spenders'
    elif row['r_score'] <= 2 and row['f_score'] <= 2:
        return 'At Risk'
    elif row['r_score'] <= 2:
        return 'Lost Customers'
    else:
        return 'Potential Loyalists'

rfm_df['rfm_segment'] = rfm_df.apply(rfm_segment, axis=1)

print("\nSegment Distribution:")
segment_counts = rfm_df['rfm_segment'].value_counts()
print(segment_counts)

In [ ]:
# Visualize RFM segments
fig, axes = plt.subplots(2, 2, figsize=(16, 12))

# Segment distribution
segment_counts.plot(kind='barh', ax=axes[0, 0], color='#2E86AB')
axes[0, 0].set_title('Customer Count by RFM Segment', fontsize=14, fontweight='bold')
axes[0, 0].set_xlabel('Number of Customers')
axes[0, 0].grid(axis='x', alpha=0.3)

# Revenue by segment
segment_revenue = rfm_df.groupby('rfm_segment')['monetary'].sum().sort_values()
segment_revenue.plot(kind='barh', ax=axes[0, 1], color='#F18F01')
axes[0, 1].set_title('Total Revenue by RFM Segment', fontsize=14, fontweight='bold')
axes[0, 1].set_xlabel('Revenue ($)')
axes[0, 1].grid(axis='x', alpha=0.3)

# RFM Score distribution
axes[1, 0].hist(rfm_df['rfm_score'], bins=15, color='#A23B72', edgecolor='black', alpha=0.7)
axes[1, 0].set_title('RFM Score Distribution', fontsize=14, fontweight='bold')
axes[1, 0].set_xlabel('RFM Score')
axes[1, 0].set_ylabel('Number of Customers')
axes[1, 0].grid(axis='y', alpha=0.3)

# Segment pie chart
axes[1, 1].pie(segment_counts.values, labels=segment_counts.index, autopct='%1.1f%%', 
               startangle=90, textprops={'fontsize': 9})
axes[1, 1].set_title('Customer Segment Distribution', fontsize=14, fontweight='bold')

plt.tight_layout()
plt.savefig('../visualizations/rfm_segments.png', dpi=300, bbox_inches='tight')
plt.show()

In [ ]:
# Segment performance analysis
segment_analysis = rfm_df.groupby('rfm_segment').agg({
    'customer_id': 'count',
    'recency': 'mean',
    'frequency': 'mean',
    'monetary': ['mean', 'sum']
}).round(2)

segment_analysis.columns = ['customer_count', 'avg_recency', 'avg_frequency', 'avg_monetary', 'total_revenue']
segment_analysis['revenue_share_%'] = (segment_analysis['total_revenue'] / segment_analysis['total_revenue'].sum() * 100).round(2)
segment_analysis = segment_analysis.sort_values('total_revenue', ascending=False)

print("\nSegment Performance Summary:")
display(segment_analysis)

## 3. K-Means Clustering

Use machine learning to identify natural customer groupings based on purchasing behavior.

In [ ]:
# Prepare data for clustering
clustering_data = rfm_df[['recency', 'frequency', 'monetary']].copy()

# Standardize the data
scaler = StandardScaler()
clustering_data_scaled = scaler.fit_transform(clustering_data)

print("Data prepared for clustering")
print(f"Shape: {clustering_data_scaled.shape}")

In [ ]:
# Find optimal number of clusters using elbow method
inertias = []
silhouette_scores = []
K_range = range(2, 11)

for k in K_range:
    kmeans = KMeans(n_clusters=k, random_state=42, n_init=10)
    kmeans.fit(clustering_data_scaled)
    inertias.append(kmeans.inertia_)
    silhouette_scores.append(silhouette_score(clustering_data_scaled, kmeans.labels_))

# Plot elbow curve and silhouette scores
fig, axes = plt.subplots(1, 2, figsize=(16, 5))

# Elbow curve
axes[0].plot(K_range, inertias, marker='o', linewidth=2, markersize=8, color='#2E86AB')
axes[0].set_xlabel('Number of Clusters (k)', fontsize=12)
axes[0].set_ylabel('Inertia', fontsize=12)
axes[0].set_title('Elbow Method For Optimal k', fontsize=14, fontweight='bold')
axes[0].grid(True, alpha=0.3)

# Silhouette scores
axes[1].plot(K_range, silhouette_scores, marker='o', linewidth=2, markersize=8, color='#F18F01')
axes[1].set_xlabel('Number of Clusters (k)', fontsize=12)
axes[1].set_ylabel('Silhouette Score', fontsize=12)
axes[1].set_title('Silhouette Score by Number of Clusters', fontsize=14, fontweight='bold')
axes[1].grid(True, alpha=0.3)

plt.tight_layout()
plt.savefig('../visualizations/optimal_clusters.png', dpi=300, bbox_inches='tight')
plt.show()

# Find optimal k
optimal_k = K_range[silhouette_scores.index(max(silhouette_scores))]
print(f"\nOptimal number of clusters: {optimal_k}")
print(f"Silhouette score: {max(silhouette_scores):.3f}")

In [ ]:
# Apply K-Means with optimal k
kmeans = KMeans(n_clusters=optimal_k, random_state=42, n_init=10)
rfm_df['cluster'] = kmeans.fit_predict(clustering_data_scaled)

print(f"Clustering complete with {optimal_k} clusters")
print("\nCluster Distribution:")
print(rfm_df['cluster'].value_counts().sort_index())

In [ ]:
# Analyze cluster characteristics
cluster_analysis = rfm_df.groupby('cluster').agg({
    'customer_id': 'count',
    'recency': 'mean',
    'frequency': 'mean',
    'monetary': ['mean', 'sum'],
    'rfm_score': 'mean'
}).round(2)

cluster_analysis.columns = ['customer_count', 'avg_recency', 'avg_frequency', 'avg_monetary', 'total_revenue', 'avg_rfm_score']
cluster_analysis = cluster_analysis.sort_values('total_revenue', ascending=False)

print("\nCluster Characteristics:")
display(cluster_analysis)

# Assign cluster names based on characteristics
def name_cluster(cluster_id):
    cluster_info = cluster_analysis.loc[cluster_id]
    
    if cluster_info['avg_monetary'] > cluster_analysis['avg_monetary'].median() and \
       cluster_info['avg_frequency'] > cluster_analysis['avg_frequency'].median():
        return 'High Value'
    elif cluster_info['avg_recency'] < cluster_analysis['avg_recency'].median():
        return 'Active'
    elif cluster_info['avg_recency'] > cluster_analysis['avg_recency'].median():
        return 'Dormant'
    else:
        return 'Medium Value'

cluster_analysis['cluster_name'] = cluster_analysis.index.map(name_cluster)
print("\nCluster Names:")
display(cluster_analysis[['cluster_name', 'customer_count', 'total_revenue']])

In [ ]:
# 3D Visualization of clusters
from mpl_toolkits.mplot3d import Axes3D

fig = plt.figure(figsize=(14, 10))
ax = fig.add_subplot(111, projection='3d')

# Plot each cluster with different color
colors = plt.cm.Set2(range(optimal_k))
for cluster_id in range(optimal_k):
    cluster_data = rfm_df[rfm_df['cluster'] == cluster_id]
    ax.scatter(cluster_data['recency'], 
               cluster_data['frequency'], 
               cluster_data['monetary'],
               c=[colors[cluster_id]], 
               label=f'Cluster {cluster_id}',
               s=50, alpha=0.6, edgecolors='black', linewidth=0.5)

ax.set_xlabel('Recency (days)', fontsize=11)
ax.set_ylabel('Frequency (purchases)', fontsize=11)
ax.set_zlabel('Monetary (revenue)', fontsize=11)
ax.set_title('Customer Segments - 3D Cluster Visualization', fontsize=14, fontweight='bold')
ax.legend(loc='upper right')

plt.savefig('../visualizations/3d_clusters.png', dpi=300, bbox_inches='tight')
plt.show()

In [ ]:
# 2D cluster visualizations
fig, axes = plt.subplots(1, 3, figsize=(18, 5))

# Recency vs Frequency
for cluster_id in range(optimal_k):
    cluster_data = rfm_df[rfm_df['cluster'] == cluster_id]
    axes[0].scatter(cluster_data['recency'], cluster_data['frequency'], 
                   label=f'Cluster {cluster_id}', alpha=0.6, s=50, edgecolors='black', linewidth=0.5)
axes[0].set_xlabel('Recency (days)')
axes[0].set_ylabel('Frequency')
axes[0].set_title('Recency vs Frequency', fontweight='bold')
axes[0].legend()
axes[0].grid(True, alpha=0.3)

# Frequency vs Monetary
for cluster_id in range(optimal_k):
    cluster_data = rfm_df[rfm_df['cluster'] == cluster_id]
    axes[1].scatter(cluster_data['frequency'], cluster_data['monetary'], 
                   label=f'Cluster {cluster_id}', alpha=0.6, s=50, edgecolors='black', linewidth=0.5)
axes[1].set_xlabel('Frequency')
axes[1].set_ylabel('Monetary ($)')
axes[1].set_title('Frequency vs Monetary', fontweight='bold')
axes[1].legend()
axes[1].grid(True, alpha=0.3)

# Recency vs Monetary
for cluster_id in range(optimal_k):
    cluster_data = rfm_df[rfm_df['cluster'] == cluster_id]
    axes[2].scatter(cluster_data['recency'], cluster_data['monetary'], 
                   label=f'Cluster {cluster_id}', alpha=0.6, s=50, edgecolors='black', linewidth=0.5)
axes[2].set_xlabel('Recency (days)')
axes[2].set_ylabel('Monetary ($)')
axes[2].set_title('Recency vs Monetary', fontweight='bold')
axes[2].legend()
axes[2].grid(True, alpha=0.3)

plt.tight_layout()
plt.savefig('../visualizations/2d_clusters.png', dpi=300, bbox_inches='tight')
plt.show()

## 4. Customer Profile Integration

In [ ]:
# Merge RFM data with customer demographics
customer_profile = rfm_df.merge(customer_df, on='customer_id', how='left')

print("Customer profiles created")
display(customer_profile.head())

In [ ]:
# Analyze demographics by RFM segment
segment_demographics = customer_profile.groupby('rfm_segment').agg({
    'age': 'mean',
    'gender': lambda x: x.value_counts().index[0] if len(x) > 0 else 'Unknown',
    'city': lambda x: x.value_counts().index[0] if len(x) > 0 else 'Unknown',
    'lifetime_value': 'mean',
    'monetary': 'mean'
}).round(2)

segment_demographics.columns = ['avg_age', 'predominant_gender', 'top_city', 'estimated_ltv', 'actual_revenue']

print("\nDemographic Profile by Segment:")
display(segment_demographics)

In [ ]:
# Analyze age distribution by cluster
fig, axes = plt.subplots(1, 2, figsize=(16, 6))

# Age by RFM segment
customer_profile.boxplot(column='age', by='rfm_segment', ax=axes[0])
axes[0].set_title('Age Distribution by RFM Segment', fontsize=14, fontweight='bold')
axes[0].set_xlabel('RFM Segment')
axes[0].set_ylabel('Age')
plt.sca(axes[0])
plt.xticks(rotation=45, ha='right')

# Gender distribution by segment
gender_segment = pd.crosstab(customer_profile['rfm_segment'], customer_profile['gender'], normalize='index') * 100
gender_segment.plot(kind='bar', stacked=True, ax=axes[1], color=['#2E86AB', '#F18F01'])
axes[1].set_title('Gender Distribution by RFM Segment (%)', fontsize=14, fontweight='bold')
axes[1].set_xlabel('RFM Segment')
axes[1].set_ylabel('Percentage')
axes[1].legend(title='Gender')
axes[1].tick_params(axis='x', rotation=45)

plt.tight_layout()
plt.savefig('../visualizations/demographic_analysis.png', dpi=300, bbox_inches='tight')
plt.show()

## 5. Marketing Recommendations

In [ ]:
# Generate actionable recommendations for each segment
recommendations = {
    'Champions': {
        'description': 'Best customers with high frequency, low recency, and high spend',
        'strategy': [
            'VIP loyalty program with exclusive rewards',
            'Early access to new products',
            'Personalized thank you campaigns',
            'Referral incentives',
            'Premium customer service'
        ],
        'priority': 'Critical'
    },
    'Loyal Customers': {
        'description': 'Regular purchasers with consistent engagement',
        'strategy': [
            'Upsell and cross-sell campaigns',
            'Loyalty point bonuses',
            'Product recommendations based on history',
            'Exclusive member discounts'
        ],
        'priority': 'High'
    },
    'High Spenders': {
        'description': 'Customers with high monetary value but lower frequency',
        'strategy': [
            'Increase purchase frequency campaigns',
            'Premium product showcases',
            'Limited-time offers',
            'Personalized product bundles'
        ],
        'priority': 'High'
    },
    'Potential Loyalists': {
        'description': 'Customers showing signs of becoming loyal',
        'strategy': [
            'Onboarding nurture campaigns',
            'Educational content about products',
            'First purchase incentives',
            'Engagement campaigns'
        ],
        'priority': 'Medium'
    },
    'At Risk': {
        'description': 'Previously active customers showing signs of churning',
        'strategy': [
            'Win-back campaigns with special offers',
            'Customer feedback surveys',
            'Re-engagement email series',
            'Personalized incentives'
        ],
        'priority': 'High'
    },
    'Lost Customers': {
        'description': 'Customers who have not purchased in a long time',
        'strategy': [
            'Aggressive win-back offers',
            'Survey to understand churn reasons',
            'Special comeback discounts',
            'Product updates and new features'
        ],
        'priority': 'Medium'
    }
}

print("="*80)
print("MARKETING RECOMMENDATIONS BY SEGMENT")
print("="*80)

for segment, info in recommendations.items():
    if segment in segment_analysis.index:
        segment_info = segment_analysis.loc[segment]
        print(f"\n{segment.upper()}")
        print("-" * 80)
        print(f"Description: {info['description']}")
        print(f"Priority: {info['priority']}")
        print(f"Customer Count: {segment_info['customer_count']:.0f}")
        print(f"Total Revenue: ${segment_info['total_revenue']:,.2f}")
        print(f"Revenue Share: {segment_info['revenue_share_%']:.1f}%")
        print("\nRecommended Strategies:")
        for i, strategy in enumerate(info['strategy'], 1):
            print(f"  {i}. {strategy}")

## 6. Export Segment Data

In [ ]:
# Save segmented customer data for marketing campaigns
customer_profile.to_csv('../data/customer_segments.csv', index=False)

print("Customer segmentation data exported to: ../data/customer_segments.csv")
print(f"Total customers segmented: {len(customer_profile):,}")
print("\nSegment counts:")
print(customer_profile['rfm_segment'].value_counts())

## Conclusion

### Key Findings:

1. **RFM Segmentation** successfully identified distinct customer groups based on behavior
2. **Machine Learning Clustering** validated and refined customer segments
3. **High-value segments** (Champions, Loyal Customers) drive significant revenue
4. **At-risk customers** require immediate retention efforts

### Business Impact:

- **Targeted Marketing**: Customize campaigns for each segment
- **Resource Optimization**: Focus efforts on high-value and at-risk customers
- **Revenue Growth**: Upsell to existing customers, re-engage dormant ones
- **Customer Retention**: Proactive churn prevention

### Next Steps:

1. Implement automated segment tracking
2. A/B test marketing strategies per segment
3. Create segment-specific dashboards
4. Monitor segment migration over time
5. Develop predictive churn models